In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/q18_rewrite/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Optimized cell 3
gb1 = lineitem.groupby("L_ORDERKEY", as_index=False, sort=False)["L_QUANTITY"].sum()
# filter orders with total quantity > 300
gb1 = gb1.loc[gb1.L_QUANTITY > 300]
# project only needed columns from orders and customer to reduce data movement
orders_sub   = orders[["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_TOTALPRICE"]]
customer_sub = customer[["C_CUSTKEY", "C_NAME"]]
# join and sort, avoiding a second groupby
total = (
    gb1
      .merge(orders_sub,   left_on="L_ORDERKEY", right_on="O_ORDERKEY")
      .merge(customer_sub, left_on="O_CUSTKEY", right_on="C_CUSTKEY")
      [["C_NAME", "C_CUSTKEY", "O_ORDERKEY", "O_ORDERDATE", "O_TOTALPRICE", "L_QUANTITY"]]
      .sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])
)

CPU times: user 47.9 ms, sys: 44 ms, total: 91.9 ms
Wall time: 91.9 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q18/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_0.pickle